Import necessary libraries

In [5]:
%matplotlib qt
%config InlineBackend.figure_formats = ['svg']

import numpy as np
from scipy.stats import (linregress, pearsonr)
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

Get .csv data for all samples from Data folder

In [6]:
samples = {}
files = [
    {
        "t = 5 nm": ["BS_5QL.csv", 5, 0],
        "t = 10 nm": ["BS_10QL.csv", 10, 0],
        "t = 16 nm": ["BS_16QL.csv", 16, 0],
        "t = 30 nm": ["BS_30QL.csv", 30, 0],
        "t = 50 nm": ["BS_50QL.csv", 50, 0],
        "t = 70 nm": ["BS_70QL.csv", 70, 0],
        "t = 100 nm": ["BS_100QL.csv", 100, 0],
    },
    {
        "x = 0": ["BS_30QL.csv", 30, 0],
        "x = 0.05": ["BIS_5%.csv", 30, 0.05],
        "x = 0.10": ["BIS_10%.csv", 30, 0.1],
        "x = 0.13": ["BIS_13%.csv", 30, 0.13],
        "x = 0.17": ["BIS_17%.csv", 30, 0.17],
        "x = 0.20": ["BIS_20%.csv", 30, 0.2]
    }
]

S_table, C_table, P_table = ({"T (°C)": [45, 55, 65, 75, 85, 95]} for _ in range(3))

for sampleset in files:
    for key in sampleset.keys():
        data = np.loadtxt("Data\\" + sampleset[key][0], delimiter=",", skiprows=1)
        data_trans = data.T
        samples[key] = data_trans
        S_table[key], C_table[key], P_table[key] = data_trans[1], data_trans[3], data_trans[5]

S_df, C_df, P_df = pd.DataFrame(S_table), pd.DataFrame(C_table), pd.DataFrame(P_table)
# S_df.to_csv("S_Table.csv", index=False)
# C_df.to_csv("C_Table.csv", index=False)
# P_df.to_csv("P_Table.csv", index=False)

Function for calculating and plotting data for either set of samples

In [7]:
def plot_samples(_samples, set_type):
    NUM_PLOTS = len(_samples)
    cm = plt.get_cmap('gist_rainbow')
    markers = ["o", "s", "^", "v", "D", "p", "*"]
    for axis in ax_list:
        axis.set_prop_cycle(color=[cm(1.*i/NUM_PLOTS) for i in range(NUM_PLOTS)], marker=markers[:NUM_PLOTS])

    x_values = []
    S_avg_list = []
    S_err_list = []
    C_avg_list = []
    C_err_list = []
    R_avg_list = []
    R_err_list = []
    P_avg_list = []
    P_err_list = []
    
    for _sample in _samples:
        T, S, errS, C, errC, P, errP = samples[_sample]
        R = 1 / (C * files[set_type - 1][_sample][1] * 10 ** -9)
        errR = errC / C * R
        P = S ** 2 * C * 10 ** -9
        errP = P * np.sqrt((2 * errS / S) ** 2 + (errC / C) ** 2)
        x_values.append(files[set_type - 1][_sample][set_type])
        S_avg = np.mean(S)
        S_std = np.sqrt(np.std(S) ** 2 + np.sum(errS ** 2) / len(errS) ** 2)
        C_avg = np.mean(C) / 1000
        C_std = np.sqrt(np.std(C) ** 2 + np.sum(errC ** 2) / len(errC) ** 2) / 1000
        R_avg = np.mean(R) / 1000
        R_std = np.sqrt(np.std(R) ** 2 + np.sum(errR ** 2) / len(errR) ** 2) / 1000
        P_avg = np.mean(P)
        P_std = np.sqrt(np.std(P) ** 2 + np.sum(errP ** 2) / len(errP) ** 2)
        S_avg_list.append(S_avg)
        S_err_list.append(S_std)
        C_avg_list.append(C_avg)
        C_err_list.append(C_std)
        R_avg_list.append(R_avg)
        R_err_list.append(R_std)
        P_avg_list.append(P_avg)
        P_err_list.append(P_std)
        
        ax_list[0].errorbar(T, S, yerr=errS, capsize=3, label=_sample, linestyle='None')
        ax_list[1].errorbar(T, R / 1000, yerr=errR/1000, capsize=3, label=_sample, linestyle='None')
        ax_list[2].errorbar(T, C / 1000, yerr=errC/1000, capsize=3, label=_sample, linestyle='None')
        ax_list[3].errorbar(T, P, yerr=errP, capsize=3, label=_sample, linestyle='None')
        

    ax_ins = [
        ax_list[0].inset_axes([0.35, 0.65, 0.3, 0.3]),
        ax_list[1].inset_axes([0.35, 0.65, 0.3, 0.3]),
        ax_list[2].inset_axes([0.35, 0.65, 0.3, 0.3]),
        ax_list[3].inset_axes([0.35, 0.65, 0.3, 0.3]),
    ]

    label_fontsize = 15
    label_padding = 2
    
    if set_type == 1:
        xlims = (0, 110)
        xlabel = r"t\;(nm)"
        xmajor_multiples = 50
        yres_multiples = 1
    else:
        xlims = (-0.01, 0.21)
        xlabel = "x"
        xmajor_multiples = 0.1
        yres_multiples = 5
    
    for ax in ax_ins:
        ax.set_xlim(xlims)
        ax.set_xlabel(f"${xlabel}$", fontsize=label_fontsize, labelpad=label_padding)
        ax.xaxis.set_major_locator(MultipleLocator(xmajor_multiples))
        ax.tick_params(axis='both', which='major', labelsize=8)
    
    ax_ins[0].errorbar(x_values, S_avg_list, yerr=S_err_list)
    ax_ins[0].set_ylim(-100, 50)
    ax_ins[0].yaxis.set_major_locator(MultipleLocator(40))
    ax_ins[0].set_ylabel(r"$S_{avg}$", fontsize=label_fontsize, labelpad=label_padding)
    
    ax_ins[1].errorbar(x_values, R_avg_list, yerr=R_err_list)
    ax_ins[1].yaxis.set_major_locator(MultipleLocator(yres_multiples))
    ax_ins[1].set_ylabel(r"$R_{avg}$", fontsize=label_fontsize, labelpad=label_padding)
    
    ax_ins[2].errorbar(x_values, C_avg_list, yerr=C_err_list)
    ax_ins[2].set_ylim(0, 110)
    ax_ins[2].yaxis.set_major_locator(MultipleLocator(50))
    ax_ins[2].set_ylabel(r"$\sigma_{avg}$", fontsize=label_fontsize, labelpad=label_padding)
    
    ax_ins[3].errorbar(x_values, P_avg_list, yerr=P_err_list)
    ax_ins[3].yaxis.set_major_locator(MultipleLocator(0.1))
    ax_ins[3].set_ylabel(r"$S^2\sigma_{avg}$", fontsize=label_fontsize, labelpad=label_padding)

    r_S, p_S = pearsonr(x_values, S_avg_list, alternative="less")
    print(r_S, p_S)
    r_P, p_P = pearsonr(x_values, P_avg_list)
    print(r_P, p_P)
        
    for _sample in _samples:
        T, S, errS, C, errC, P, errP = samples[_sample]
        R = 1 / (C * files[set_type - 1][_sample][1] * 10 ** -9)
        P = S ** 2 * C * 10 ** -9
        
        mS, bS, rS, pS, std_errS = linregress(T, S)
        mC, bC, rC, pC, std_errC = linregress(T, C / 1000)
        mR, bR, rR, pR, std_errR = linregress(T, R / 1000)
        mP, bP, rP, pP, std_errP = linregress(T, P)
        
        ax_list[0].plot(T, mS * T + bS, color="black", marker="None")
        ax_list[1].plot(T, mR * T + bR, color="black", marker="None")
        ax_list[2].plot(T, mC * T + bC, color="black", marker="None")
        ax_list[3].plot(T, mP * T + bP, color="black", marker="None")

Mode 1: Thickness dependence, Mode 2: Composition dependence

In [8]:
mode = 1

Configure formatting and create charts

In [12]:
plt.style.use("classic")
fig, axes = plt.subplots(2, 2)
ax_list = [
    axes[0, 0],
    axes[0, 1],
    axes[1, 0],
    axes[1, 1]
]
fig.tight_layout()
fig.set_size_inches(12, 10)
plt.subplots_adjust(left=0.11, right=0.95, top=0.95, bottom=0.08, wspace=0.3, hspace=0.3)

if mode == 1:
    plot_samples(list(files[0].keys()), 1)
    res_ylim = 7
else:
    plot_samples(list(files[1].keys()), 2)
    res_ylim = 30

for axis in ax_list:
    axis.legend(loc="upper right", fontsize=9, numpoints=1)

ax_list[0].set_title(r"$S$ vs. $T$", fontsize="20")
ax_list[0].set_xlabel(r"$T\;(\mathrm{\degree C})$", fontsize=18)
ax_list[0].set_ylabel(r"$S\;\left(\mathrm{\frac{\mu V}{K}}\right)$", fontsize=18)
ax_list[0].set_ylim(-100, 50)
ax_list[0].text(0.03, 0.92, "(a)", fontsize=18, transform=ax_list[0].transAxes)

ax_list[1].set_title(r"$R_S$ vs. $T$", fontsize="20")
ax_list[1].set_xlabel(r"$T\;(\mathrm{\degree C})$", fontsize=18)
ax_list[1].set_ylabel(r"$R_S\;\left(\mathrm{\frac{k\Omega}{□}}\right)$", fontsize=18)
ax_list[1].set_ylim(0, res_ylim)
ax_list[1].text(0.03, 0.92, "(b)", fontsize=18, transform=ax_list[1].transAxes)

ax_list[2].set_title(r"$\sigma$ vs. $T$", fontsize="20")
ax_list[2].set_xlabel(r"$T\;(\mathrm{\degree C})$", fontsize=18)
ax_list[2].set_ylabel(r"$\sigma\;\left(\mathrm{\frac{kS}{m}}\right)$", fontsize=18)
ax_list[2].set_ylim(0, 200)
ax_list[2].text(0.03, 0.92, "(c)", fontsize=18, transform=ax_list[2].transAxes)

ax_list[3].set_title(r"$S^2\sigma$ vs. $T$", fontsize="20")
ax_list[3].set_xlabel(r"$T\;(\mathrm{\degree C})$", fontsize=18)
ax_list[3].set_ylabel(r"$S^2\sigma\;\left(\mathrm{\frac{mW}{m\cdot K^2}}\right)$", fontsize=18)
ax_list[3].set_ylim(0, 0.6)
ax_list[3].text(0.03, 0.92, "(d)", fontsize=18, transform=ax_list[3].transAxes)

-0.8444488040782806 0.008412458022391458
0.4435288110624968 0.3188624993961005


Text(0.03, 0.92, '(d)')

Save figures as images

In [13]:
if mode == 1:
    plt.savefig("Figure_2_2x2_HR.png", dpi=300)
else:
    plt.savefig("Figure_1_2x2_HR.png", dpi=300)